# V1-15: Thinking 비활성화 + Schema Grounding 평가

14번에서 기본 thinking 상태의 동일 구조화 호출은 장시간 완료되지 않았다. 이 노트북은 `reasoning=False`와 JSON schema의 프롬프트 명시를 함께 적용해, 호출 안정성·구조화 출력·기업 후보 회수를 평가한다.

> 14번의 기본 thinking 조건은 기준선으로 보존한다. 이 노트북은 모델·temperature·평가 사례를 유지하고, thinking과 schema 전달 방식만 바꾼 후보다.

## 비교 대상과 해석 한계

- **A. deterministic 기준선**: 원문 표현으로 OpenDART exact + fuzzy 후보를 만든다.
- **B. V15 후보**: 고유 raw exact가 아닌 입력에서 `reasoning=False` LLM이 검색어 가설을 JSON으로 반환하고, 코드가 OpenDART 후보를 다시 탐색한다.

14번은 기본 thinking 호출이 완료되지 않아 후보 품질의 공정 비교 기준으로 쓰지 않는다. 이 노트북은 먼저 B가 실용적인 시간 안에 안정적으로 동작하는지와 A 대비 후보 회수 이득이 있는지를 확인한다.

이번 평가는 `reasoning=False + schema grounding`의 **결합 조건** 평가다. 두 변경 중 어느 하나만의 효과를 분리해 결론 내리지 않는다. smoke 호출 이후의 평가 지연시간은 warm-run 기준으로 해석한다.

## 평가 지표

- Candidate Recall / LLM Candidate Recall / LLM Expansion Contribution
- Abstention Accuracy
- Structured-output Success Rate / LLM Search-term Validity
- LLM Call Rate / p50·p95 지연시간

품질·안전성을 우선으로 해석한다. 다만 14번처럼 장시간 완료되지 않는 현상은 단순한 속도 차이가 아니라 운영 안정성 문제이므로, B가 구조화 호출을 완료하지 못하면 미채택한다.

In [ ]:
import io
from datetime import datetime, timezone
from importlib.metadata import version
import json
from math import ceil
import os
from pathlib import Path
import re
from statistics import median
import sys
import time
from typing import Literal
import xml.etree.ElementTree as ET
import zipfile

from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from rapidfuzz import fuzz, process

In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
MODEL_NAME = os.environ.get('OLLAMA_MODEL', 'qwen3.6:27b')
TEMPERATURE = 0
PROMPT_VERSION = 'entity-search-plan-v3-reasoning-off-schema-grounded'
REQUEST_TIMEOUT_S = 300

print(f'모델: {MODEL_NAME} / temperature={TEMPERATURE}')
print(f'reasoning=False / 요청 timeout={REQUEST_TIMEOUT_S}초 / runner 옵션은 지정하지 않음')

In [ ]:
sys.path.insert(0, str(project_root / 'src'))

from dart.client import CORP_CODE_URL, _get_bytes


def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    zip_stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(zip_stream):
        raise RuntimeError(zip_bytes.decode('utf-8', errors='replace'))
    with zipfile.ZipFile(zip_stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])
    root = ET.fromstring(xml_bytes)
    return [{child.tag: child.text or '' for child in item} for item in root.findall('list')]


corp_records = load_corp_records(api_key)
retrieved_at = datetime.now().astimezone().isoformat(timespec='seconds')
print(f'OpenDART 기업코드 조회 시각: {retrieved_at}')
print(f'전체 법인 레코드 수: {len(corp_records):,}')

In [ ]:
def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())


def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(values.values()) for key, values in index.items()}


name_index = build_name_index(corp_records)
name_keys = list(name_index)

TARGETS = {
    '삼성전자': ('삼성전자', '005930'), 'SK하이닉스': ('SK하이닉스', '000660'),
    'LG전자': ('LG전자', '066570'), '현대자동차': ('현대자동차', '005380'),
    'NAVER': ('NAVER', '035420'), '카카오': ('카카오', '035720'),
    'POSCO홀딩스': ('POSCO홀딩스', '005490'), '삼성바이오로직스': ('삼성바이오로직스', '207940'),
    '셀트리온': ('셀트리온', '068270'), '기아': ('기아', '000270'),
}
target_records = {}
for key, (official_name, stock_code) in TARGETS.items():
    matches = [record for record in corp_records if normalize_name(record['corp_name']) == normalize_name(official_name) and record['stock_code'].strip() == stock_code]
    if len(matches) != 1:
        raise AssertionError(f'{key}: OpenDART 기준 법인 검증 실패 ({len(matches)}건)')
    target_records[key] = matches[0]

print(f'정규화 이름 키 수: {len(name_keys):,}')

In [ ]:
CASES = [
    ('삼성전자 2024년 매출을 알려줘.', '삼성전자', ['삼성전자'], 'skip'),
    ('삼성 전자 실적을 요약해줘.', '삼성 전자', ['삼성전자'], 'skip'),
    ('삼전의 최근 매출 추세를 보여줘.', '삼전', ['삼성전자'], 'candidate'),
    ('삼성전쟈 영업이익을 알려줘.', '삼성전쟈', ['삼성전자'], 'candidate'),
    ('SK하이닉스의 사업보고서 기준 매출은?', 'SK하이닉스', ['SK하이닉스'], 'skip'),
    ('sk 하이닉스 실적을 알려줘.', 'sk 하이닉스', ['SK하이닉스'], 'skip'),
    ('하이닉스 매출 추세를 보고 싶어.', '하이닉스', ['SK하이닉스'], 'candidate'),
    ('하닉 실적을 정리해줘.', '하닉', ['SK하이닉스'], 'candidate'),
    ('LG전자 매출을 알려줘.', 'LG전자', ['LG전자'], 'skip'),
    ('엘지전자 영업이익 추이를 알려줘.', '엘지전자', ['LG전자'], 'candidate'),
    ('현대자동차의 최근 실적은?', '현대자동차', ['현대자동차'], 'skip'),
    ('현대차 매출을 알려줘.', '현대차', ['현대자동차'], 'candidate'),
    ('현차 최근 매출 추세는?', '현차', ['현대자동차'], 'candidate'),
    ('NAVER의 매출을 알려줘.', 'NAVER', ['NAVER'], 'skip'),
    ('네이버 영업이익을 알려줘.', '네이버', ['NAVER'], 'candidate'),
    ('카카오의 매출을 알려줘.', '카카오', ['카카오'], 'skip'),
    ('POSCO홀딩스 실적을 분석해줘.', 'POSCO홀딩스', ['POSCO홀딩스'], 'skip'),
    ('포스코홀딩스 영업이익을 보여줘.', '포스코홀딩스', ['POSCO홀딩스'], 'candidate'),
    ('삼바의 최근 실적은?', '삼바', ['삼성바이오로직스'], 'candidate'),
    ('셀트 매출 추세를 알려줘.', '셀트', ['셀트리온'], 'candidate'),
    ('기아차의 영업이익을 알려줘.', '기아차', ['기아'], 'candidate'),
    ('엘지 계열사의 실적을 알려줘.', '엘지', [], 'abstain'),
    ('현대 실적을 알려줘.', '현대', [], 'abstain'),
    ('감자우주물산2026의 매출을 알려줘.', '감자우주물산2026', [], 'abstain'),
    ('BLUEORBITVENTURES2049의 실적을 알려줘.', 'BLUEORBITVENTURES2049', [], 'abstain'),
    ('QXZ테스트법인9999 재무정보를 알려줘.', 'QXZ테스트법인9999', [], 'abstain'),
]
TEST_CASES = [
    {'질문': question, '원문 기업 표현': raw_name, '기대 대상': targets, 'LLM 기대 행동': expectation}
    for question, raw_name, targets, expectation in CASES
]
print(f'전체 사례: {len(TEST_CASES)}개')

In [ ]:
FUZZY_KEY_LIMIT = 50
FUZZY_RESULT_LIMIT = 5


def exact_records(query):
    return name_index.get(normalize_name(query), [])


def fuzzy_records(query):
    scored_keys = process.extract(normalize_name(query), name_keys, scorer=fuzz.ratio, limit=FUZZY_KEY_LIMIT)
    best_by_code = {}
    for key, score, _ in scored_keys:
        for record in name_index[key]:
            if record['corp_code'] not in best_by_code or score > best_by_code[record['corp_code']][1]:
                best_by_code[record['corp_code']] = (record, score)
    return sorted(best_by_code.values(), key=lambda item: (item[1], bool(item[0]['stock_code'].strip()), item[0]['modify_date']), reverse=True)[:FUZZY_RESULT_LIMIT]


def union_candidates(query, source_prefix):
    candidates = {}
    for record in exact_records(query):
        candidates[record['corp_code']] = {'record': record, 'sources': {f'{source_prefix}_exact'}}
    for record, score in fuzzy_records(query):
        current = candidates.setdefault(record['corp_code'], {'record': record, 'sources': set()})
        current['sources'].add(f'{source_prefix}_fuzzy')
    return candidates


def merge_candidates(*candidate_sets):
    merged = {}
    for candidate_set in candidate_sets:
        for corp_code, candidate in candidate_set.items():
            current = merged.setdefault(corp_code, {'record': candidate['record'], 'sources': set()})
            current['sources'].update(candidate['sources'])
    return merged


def needs_llm(raw_name):
    return len(exact_records(raw_name)) != 1

print('실제 LLM 호출 대상:', sum(needs_llm(case['원문 기업 표현']) for case in TEST_CASES))

In [ ]:
class EntitySearchPlan(BaseModel):
    decision: Literal['candidate', 'ambiguous', 'unknown'] = Field(
        description='검색 가설 제안, 포괄·다의적 표현, 또는 해석 불가 상태'
    )
    search_terms: list[str] = Field(
        default_factory=list,
        max_length=2,
        description='OpenDART에서 검증할 정식 기업명 검색 가설. corp_code·종목코드·설명 문장은 금지.'
    )

BASE_SYSTEM_PROMPT = '''
당신은 한국 기업 리서치 시스템의 기업명 검색 계획기다.
프로그램은 원문 기업 표현을 OpenDART에서 고유하게 exact 확정하지 못했을 때만 당신을 호출한다.

원문이 명백한 약칭, 한글·영문 표기 차이, 또는 가벼운 철자 오타로 보이면 decision='candidate'와 함께
정식 기업명으로 보이는 검색 가설을 최대 2개 제안한다. 완전한 확신이 없어도 검색 가설은 제안할 수 있다.
이 가설은 정답 확정이 아니며, 뒤의 프로그램이 OpenDART 기업 목록으로 검증하고 필요하면 사용자에게 재질문한다.

원문이 포괄적인 그룹명, 여러 기업을 뜻할 수 있는 표현, 또는 실제 기업으로 해석할 근거가 없는 표현이면
decision='ambiguous' 또는 decision='unknown'을 반환하고 search_terms=[]로 둔다.

사용자가 언급하지 않은 계열사·경쟁사·관련 기업을 추가하지 않는다.
corp_code, 종목코드, 투자 의견, 설명 문장을 반환하지 않는다.
'''

SCHEMA_JSON = json.dumps(EntitySearchPlan.model_json_schema(), ensure_ascii=False)
SYSTEM_PROMPT = (
    BASE_SYSTEM_PROMPT
    + '\n반드시 설명 없이 유효한 JSON 객체만 반환한다. 아래 JSON Schema를 반드시 만족한다.\nJSON Schema: '
    + SCHEMA_JSON
)

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=TEMPERATURE,
    reasoning=False,
    client_kwargs={'timeout': REQUEST_TIMEOUT_S},
)
planner = llm.with_structured_output(EntitySearchPlan, include_raw=True)

RUN_CONFIG = {
    'run_at_utc': datetime.now(timezone.utc).isoformat(),
    'model': MODEL_NAME, 'temperature': TEMPERATURE, 'reasoning': False,
    'schema_grounding': True, 'request_timeout_s': REQUEST_TIMEOUT_S, 'prompt_version': PROMPT_VERSION,
    'langchain_version': version('langchain'),
    'langchain_ollama_version': version('langchain-ollama'),
}
RUN_CONFIG

In [ ]:
def build_messages(question, raw_name):
    return [
        ('system', SYSTEM_PROMPT),
        ('human', f'사용자 질문: {question}\n원문 기업 표현: {raw_name}'),
    ]


def raw_terms(plan):
    return list(plan.search_terms) if plan is not None else []


def clean_terms(plan):
    if plan is None:
        return [], []
    if plan.decision != 'candidate':
        return [], raw_terms(plan)
    terms, rejected, seen = [], [], set()
    for term in plan.search_terms:
        term = term.strip()
        key = normalize_name(term)
        if not term or len(term) > 80 or not key or key.isdigit() or key in seen:
            rejected.append(term)
            continue
        seen.add(key)
        terms.append(term)
    return terms[:2], rejected


print('[사전 호출 1/1] 하닉 / reasoning=False / 호출 시작', flush=True)
smoke_started_at = time.perf_counter()
try:
    smoke = planner.invoke(build_messages('하닉 실적을 정리해줘.', '하닉'))
    print(f'[사전 호출 1/1] 완료 / {(time.perf_counter() - smoke_started_at) * 1_000:,.0f}ms', flush=True)
    print('파싱 결과:', smoke['parsed'])
    print('파싱 오류:', smoke['parsing_error'])
except Exception as error:
    print(f'[사전 호출 1/1] 실패 / {(time.perf_counter() - smoke_started_at) * 1_000:,.0f}ms', flush=True)
    print(f'{type(error).__name__}: {error}')

## B 후보 실행

고유 raw exact가 아닌 입력에만 reasoning-disabled LLM을 한 번씩 호출한다. 진행 출력은 현재 호출 번호와 경과시간을 보여 준다.

In [ ]:
rows = []
llm_total = sum(needs_llm(case['원문 기업 표현']) for case in TEST_CASES)
llm_number = 0
print(f'전체 평가 시작: LLM 호출 {llm_total}회', flush=True)

for case_number, case in enumerate(TEST_CASES, start=1):
    raw_name = case['원문 기업 표현']
    baseline = union_candidates(raw_name, 'raw')
    llm_called = needs_llm(raw_name)
    plan, error = None, None
    raw_search_terms, proposed_terms, rejected_terms, llm_latency_ms = [], [], [], None
    llm_candidates = {}

    if llm_called:
        llm_number += 1
        print(f'[LLM {llm_number}/{llm_total}] {raw_name!r} / 호출 시작', flush=True)
        started_at = time.perf_counter()
        try:
            result = planner.invoke(build_messages(case['질문'], raw_name))
            plan, error = result['parsed'], result['parsing_error']
            raw_search_terms = raw_terms(plan)
            proposed_terms, rejected_terms = clean_terms(plan)
            llm_candidates = merge_candidates(*(union_candidates(term, 'llm') for term in proposed_terms))
        except Exception as exc:
            error = f'{type(exc).__name__}: {exc}'
        finally:
            llm_latency_ms = (time.perf_counter() - started_at) * 1_000
            status = '완료' if error is None and plan is not None else '실패'
            print(f'[LLM {llm_number}/{llm_total}] {status} / {llm_latency_ms:,.0f}ms', flush=True)
    else:
        print(f'[건너뜀] {raw_name!r} / 고유 raw exact', flush=True)

    expanded = merge_candidates(baseline, llm_candidates)
    expected_codes = {target_records[key]['corp_code'] for key in case['기대 대상']}
    baseline_codes, llm_codes, expanded_codes = set(baseline), set(llm_candidates), set(expanded)
    term_validity = [bool(term.strip()) and bool(exact_records(term.strip())) for term in raw_search_terms]
    is_safe_abstention = plan is not None and plan.decision in {'ambiguous', 'unknown'} and not raw_search_terms
    rows.append({
        **case, 'LLM 호출': llm_called, 'LLM 결정': plan.decision if plan else None,
        'LLM 원본 검색어': raw_search_terms, '실제 탐색 검색어': proposed_terms, '거부된 검색어': rejected_terms,
        '구조화 출력 성공': None if not llm_called else error is None and plan is not None,
        '원본 검색어 exact 유효 여부': term_validity,
        '기준선 정답 포함': bool(expected_codes & baseline_codes) if expected_codes else None,
        'LLM 정답 포함': bool(expected_codes & llm_codes) if expected_codes else None,
        '확장 후 정답 포함': bool(expected_codes & expanded_codes) if expected_codes else None,
        'LLM 신규 기여': bool((expected_codes & llm_codes) - baseline_codes) if expected_codes else None,
        '안전 보류': is_safe_abstention if case['LLM 기대 행동'] == 'abstain' else None,
        'LLM 호출 시간(ms)': llm_latency_ms, '오류': str(error) if error else None,
    })

print(f'전체 평가 완료: 사례 {len(rows)}개, LLM 호출 {llm_number}회', flush=True)
len(rows)

In [ ]:
positive_rows = [row for row in rows if row['기대 대상']]
llm_positive_rows = [row for row in rows if row['LLM 호출'] and row['LLM 기대 행동'] == 'candidate']
abstain_rows = [row for row in rows if row['LLM 기대 행동'] == 'abstain']
llm_rows = [row for row in rows if row['LLM 호출']]
validity_values = [value for row in llm_rows for value in row['원본 검색어 exact 유효 여부']]
latencies = [row['LLM 호출 시간(ms)'] for row in llm_rows if row['LLM 호출 시간(ms)'] is not None]

def p95(values):
    return sorted(values)[ceil(len(values) * 0.95) - 1] if values else None

summary = {
    '평가 사례 수': len(rows),
    'A. 기준선 Candidate Recall': sum(row['기준선 정답 포함'] for row in positive_rows) / len(positive_rows),
    'B. reasoning-off Candidate Recall': sum(row['확장 후 정답 포함'] for row in positive_rows) / len(positive_rows),
    'B. LLM Candidate Recall': sum(row['LLM 정답 포함'] for row in llm_positive_rows) / len(llm_positive_rows),
    'B. LLM Expansion Contribution': sum(row['LLM 신규 기여'] for row in positive_rows) / len(positive_rows),
    'B. Abstention Accuracy': sum(row['안전 보류'] for row in abstain_rows) / len(abstain_rows),
    'B. Structured-output Success Rate': sum(row['구조화 출력 성공'] for row in llm_rows) / len(llm_rows),
    'B. Search-term Validity': sum(validity_values) / len(validity_values) if validity_values else None,
    'B. LLM Call Rate': len(llm_rows) / len(rows),
    'B. LLM p50(ms)': median(latencies) if latencies else None,
    'B. LLM p95(ms)': p95(latencies),
}
summary

In [ ]:
import pandas as pd

columns = [
    '원문 기업 표현', 'LLM 기대 행동', 'LLM 호출', 'LLM 결정', 'LLM 원본 검색어', '실제 탐색 검색어',
    '기준선 정답 포함', 'LLM 정답 포함', '확장 후 정답 포함', 'LLM 신규 기여',
    '안전 보류', '원본 검색어 exact 유효 여부', 'LLM 호출 시간(ms)', '오류',
]
display(pd.DataFrame(rows)[columns])

## 해석 기준

- B의 Structured-output Success Rate가 낮으면 reasoning-off + schema grounding 후보는 형식 신뢰성 부족으로 미채택한다.
- B의 Candidate Recall 또는 LLM Expansion Contribution이 A보다 높아야 LLM 검색어 확장을 유지할 근거가 생긴다.
- B의 Abstention Accuracy가 낮으면 무관한 기업 후보를 만드는 위험이 있으므로 미채택한다.
- 지연시간은 참고 지표지만, 개별 호출이 완료되지 않는 현상은 운영 안정성 실패로 별도 기록한다. 요청은 300초에 timeout으로 실패 행에 기록한다.

실행 결과를 저장한 뒤 알려주면, 15번 결과의 채택·미채택 근거를 HTML 보고서로 작성한다.